In [1]:
import os

proj = r"C:\Users\kgeet\bbim707-hadoop"      # project folder for the Hadoop setup
os.makedirs(proj, exist_ok=True)

compose = """services:
  namenode:
    image: bde2020/hadoop-namenode:2.0.0-hadoop3.2.1-java8
    container_name: namenode
    restart: always
    ports:
      - "9870:9870"        # HDFS web dashboard
      - "9000:9000"        # HDFS filesystem port
    volumes:
      - hadoop_namenode:/hadoop/dfs/name
      - C:/Users/kgeet/data:/data    # your exported CSVs, visible inside the container at /data
    environment:
      - CLUSTER_NAME=bbim707
    env_file:
      - ./hadoop.env

  datanode:
    image: bde2020/hadoop-datanode:2.0.0-hadoop3.2.1-java8
    container_name: datanode
    restart: always
    volumes:
      - hadoop_datanode:/hadoop/dfs/data
    environment:
      SERVICE_PRECONDITION: "namenode:9870"   # wait for the namenode before starting
    env_file:
      - ./hadoop.env

volumes:
  hadoop_namenode:
  hadoop_datanode:
"""

hadoop_env = """CORE_CONF_fs_defaultFS=hdfs://namenode:9000
CORE_CONF_hadoop_http_staticuser_user=root
HDFS_CONF_dfs_webhdfs_enabled=true
HDFS_CONF_dfs_permissions_enabled=false
HDFS_CONF_dfs_namenode_datanode_registration_ip___hostname___check=false
"""

# newline="\n" forces Linux line endings - important, since the container reads these as Linux files
with open(os.path.join(proj, "docker-compose.yml"), "w", newline="\n") as f:
    f.write(compose)
with open(os.path.join(proj, "hadoop.env"), "w", newline="\n") as f:
    f.write(hadoop_env)

print("Created in", proj)
for fn in os.listdir(proj):
    print("  ", fn)

Created in C:\Users\kgeet\bbim707-hadoop
   docker-compose.yml
   hadoop-hive.env
   hadoop.env


In [2]:
import os
proj = r"C:\Users\kgeet\bbim707-hadoop"

compose = """services:
  namenode:
    image: bde2020/hadoop-namenode:2.0.0-hadoop3.2.1-java8
    container_name: namenode
    restart: always
    ports:
      - "9870:9870"
      - "9000:9000"
    volumes:
      - hadoop_namenode:/hadoop/dfs/name
      - C:/Users/kgeet/data:/data
    environment:
      - CLUSTER_NAME=bbim707
    env_file:
      - ./hadoop.env

  datanode:
    image: bde2020/hadoop-datanode:2.0.0-hadoop3.2.1-java8
    container_name: datanode
    restart: always
    volumes:
      - hadoop_datanode:/hadoop/dfs/data
    environment:
      SERVICE_PRECONDITION: "namenode:9870"
    env_file:
      - ./hadoop.env

  datanode2:
    image: bde2020/hadoop-datanode:2.0.0-hadoop3.2.1-java8
    container_name: datanode2
    restart: always
    volumes:
      - hadoop_datanode2:/hadoop/dfs/data
    environment:
      SERVICE_PRECONDITION: "namenode:9870"
    env_file:
      - ./hadoop.env

volumes:
  hadoop_namenode:
  hadoop_datanode:
  hadoop_datanode2:
"""
with open(os.path.join(proj, "docker-compose.yml"), "w", newline="\n") as f:
    f.write(compose)
print("compose updated with datanode2")

compose updated with datanode2


# Write the Hive config and updated compose

In [3]:
import os
proj = r"C:\Users\kgeet\bbim707-hadoop"

# Hive settings - every HDFS reference uses port 9000 to match the running namenode
hadoop_hive_env = """HIVE_SITE_CONF_javax_jdo_option_ConnectionURL=jdbc:postgresql://hive-metastore-postgresql/metastore
HIVE_SITE_CONF_javax_jdo_option_ConnectionDriverName=org.postgresql.Driver
HIVE_SITE_CONF_javax_jdo_option_ConnectionUserName=hive
HIVE_SITE_CONF_javax_jdo_option_ConnectionPassword=hive
HIVE_SITE_CONF_datanucleus_autoCreateSchema=false
HIVE_SITE_CONF_hive_metastore_uris=thrift://hive-metastore:9083
HIVE_SITE_CONF_hive_metastore_warehouse_dir=hdfs://namenode:9000/user/hive/warehouse
CORE_CONF_fs_defaultFS=hdfs://namenode:9000
CORE_CONF_hadoop_http_staticuser_user=root
HDFS_CONF_dfs_webhdfs_enabled=true
HDFS_CONF_dfs_permissions_enabled=false
HDFS_CONF_dfs_namenode_datanode_registration_ip___hostname___check=false
"""

compose = """services:
  namenode:
    image: bde2020/hadoop-namenode:2.0.0-hadoop3.2.1-java8
    container_name: namenode
    restart: always
    ports:
      - "9870:9870"
      - "9000:9000"
    volumes:
      - hadoop_namenode:/hadoop/dfs/name
      - C:/Users/kgeet/data:/data
    environment:
      - CLUSTER_NAME=bbim707
    env_file:
      - ./hadoop.env

  datanode:
    image: bde2020/hadoop-datanode:2.0.0-hadoop3.2.1-java8
    container_name: datanode
    restart: always
    volumes:
      - hadoop_datanode:/hadoop/dfs/data
    environment:
      SERVICE_PRECONDITION: "namenode:9870"
    env_file:
      - ./hadoop.env

  hive-metastore-postgresql:
    image: bde2020/hive-metastore-postgresql:2.3.0
    container_name: hive-metastore-postgresql

  hive-metastore:
    image: bde2020/hive:2.3.2-postgresql-metastore
    container_name: hive-metastore
    env_file:
      - ./hadoop-hive.env
    command: /opt/hive/bin/hive --service metastore
    environment:
      SERVICE_PRECONDITION: "namenode:9000 datanode:9864 hive-metastore-postgresql:5432"
    ports:
      - "9083:9083"

  hive-server:
    image: bde2020/hive:2.3.2-postgresql-metastore
    container_name: hive-server
    env_file:
      - ./hadoop-hive.env
    environment:
      HIVE_CORE_CONF_javax_jdo_option_ConnectionURL: "jdbc:postgresql://hive-metastore-postgresql/metastore"
      SERVICE_PRECONDITION: "hive-metastore:9083"
    ports:
      - "10000:10000"

volumes:
  hadoop_namenode:
  hadoop_datanode:
"""

with open(os.path.join(proj, "hadoop-hive.env"), "w", newline="\n") as f:
    f.write(hadoop_hive_env)
with open(os.path.join(proj, "docker-compose.yml"), "w", newline="\n") as f:
    f.write(compose)
print("Wrote hadoop-hive.env and updated docker-compose.yml (Hive added, datanode2 removed)")

Wrote hadoop-hive.env and updated docker-compose.yml (Hive added, datanode2 removed)


In [4]:
# one-time install of the MongoDB driver
!pip install pymongo        

import pandas as pd, json
from pymongo import MongoClient

# Read the cleaned country reference and convert to clean documents
# json round-trip gives native types and turns NaN into null
countries = pd.read_csv(r"C:\Users\kgeet\data\countries_clean.csv")
records = json.loads(countries.to_json(orient="records"))

# Connect to the MongoDB container 
client = MongoClient("mongodb://localhost:27017/")
db   = client["retail"]        # database
coll = db["countries"]         # collection 

coll.drop()                    # clear any previous run to prevent duplication
coll.insert_many(records)      # store one document per country
print("Documents stored:", coll.count_documents({}))

Documents stored: 227


In [5]:
# Query 1 (filter, projection, sort): high-income Western European markets
print("High-income Western European markets:")
for doc in coll.find(
        {"region": "Western Europe", "income_group": "High"},
        {"_id": 0, "country": 1, "gdp_per_capita": 1}).sort("gdp_per_capita", -1):
    print(" ", doc)

# Query 2 (aggregation pipeline): count of countries per income group
print("\nCountries per income group:")
for doc in coll.aggregate([
        {"$group": {"_id": "$income_group", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}}]):
    print(" ", doc)

High-income Western European markets:
  {'country': 'Luxembourg', 'gdp_per_capita': 55100.0}
  {'country': 'Norway', 'gdp_per_capita': 37800.0}
  {'country': 'San Marino', 'gdp_per_capita': 34600.0}
  {'country': 'Switzerland', 'gdp_per_capita': 32700.0}
  {'country': 'Denmark', 'gdp_per_capita': 31100.0}
  {'country': 'Iceland', 'gdp_per_capita': 30900.0}
  {'country': 'Austria', 'gdp_per_capita': 30000.0}
  {'country': 'Ireland', 'gdp_per_capita': 29600.0}
  {'country': 'Belgium', 'gdp_per_capita': 29100.0}
  {'country': 'Netherlands', 'gdp_per_capita': 28600.0}
  {'country': 'United Kingdom', 'gdp_per_capita': 27700.0}
  {'country': 'France', 'gdp_per_capita': 27600.0}
  {'country': 'Germany', 'gdp_per_capita': 27600.0}
  {'country': 'Finland', 'gdp_per_capita': 27400.0}
  {'country': 'Monaco', 'gdp_per_capita': 27000.0}
  {'country': 'Sweden', 'gdp_per_capita': 26800.0}
  {'country': 'Italy', 'gdp_per_capita': 26700.0}

Countries per income group:
  {'_id': 'Lower-middle', 'count':

# Kafka setup

In [6]:
import os
proj = r"C:\Users\kgeet\bbim707-streaming"
os.makedirs(proj, exist_ok=True)

compose = """services:
  kafka:
    image: apache/kafka:3.7.0
    container_name: kafka
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:29092,CONTROLLER://0.0.0.0:9093,EXTERNAL://0.0.0.0:9092
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:29092,EXTERNAL://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT,EXTERNAL:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
"""
with open(os.path.join(proj, "docker-compose.yml"), "w", newline="\n") as f:
    f.write(compose)
print("Wrote streaming compose to", proj)

Wrote streaming compose to C:\Users\kgeet\bbim707-streaming


In [7]:
!pip install confluent-kafka

   ---------------------------------------- 0.0/4.5 MB ? eta -:--:--
   --------- ------------------------------ 1.0/4.5 MB 9.1 MB/s eta 0:00:01
   ---------------- ----------------------- 1.8/4.5 MB 4.1 MB/s eta 0:00:01
   ---------------------------- ----------- 3.1/4.5 MB 5.6 MB/s eta 0:00:01
   ------------------------------------- -- 4.2/4.5 MB 5.6 MB/s eta 0:00:01
   ------------------------------------- -- 4.2/4.5 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 4.5/4.5 MB 3.8 MB/s  0:00:01


In [8]:
import pandas as pd, json, time
from confluent_kafka import Producer

# Load the transactions and take a 5,000-row sample to simulate a live feed
df = pd.read_csv(r"C:\Users\kgeet\data\retail_enriched.csv")
stream_df = df.sample(n=5000, random_state=42)

# Connect to Kafka via the external listener
producer = Producer({"bootstrap.servers": "localhost:9092"})

# Stream each transaction as one JSON message into the topic
sent = 0
for record in json.loads(stream_df.to_json(orient="records")):
    producer.produce("retail-stream", value=json.dumps(record).encode("utf-8"))
    sent += 1
    if sent % 1000 == 0:
        producer.flush()                 # push out the batch
        print(f"Sent {sent} messages...")
        time.sleep(0.5)                  # small pause to mimic a real-time trickle

producer.flush()                         
print(f"Done. Total messages sent: {sent}")

Sent 1000 messages...
Sent 2000 messages...
Sent 3000 messages...
Sent 4000 messages...
Sent 5000 messages...
Done. Total messages sent: 5000


In [10]:
import os
proj = r"C:\Users\kgeet\bbim707-streaming"
os.makedirs(os.path.join(proj, "scripts"), exist_ok=True)

compose = """services:
  kafka:
    image: apache/kafka:3.7.0
    container_name: kafka
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:29092,CONTROLLER://0.0.0.0:9093,EXTERNAL://0.0.0.0:9092
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:29092,EXTERNAL://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT,EXTERNAL:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0

  spark:
    image: apache/spark:latest
    container_name: spark
    entrypoint: ["sleep", "infinity"]
    volumes:
      - C:/Users/kgeet/bbim707-streaming/scripts:/opt/scripts
"""
with open(os.path.join(proj, "docker-compose.yml"), "w", newline="\n") as f:
    f.write(compose)
print("Compose updated to use apache/spark:latest")

Compose updated to use apache/spark:latest


In [11]:
# Streaming cluster
script = '''from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, count, avg, round as sround
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

spark = SparkSession.builder.appName("RetailStreamClustering").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Fields we read from each JSON message (numeric features + country for context)
schema = (StructType()
          .add("quantity", LongType())
          .add("price", DoubleType())
          .add("revenue", DoubleType())
          .add("country", StringType()))

# Read the live Kafka stream via the internal listener (kafka:29092)
raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", "kafka:29092")
       .option("subscribe", "retail-stream")
       .option("startingOffsets", "earliest")
       .load())

# Kafka value is bytes -> string -> parse JSON into typed columns
parsed = (raw.selectExpr("CAST(value AS STRING) AS json")
          .select(from_json(col("json"), schema).alias("d"))
          .select("d.*"))

# Cluster each micro-batch of transactions as it arrives
def cluster_batch(batch_df, batch_id):
    n = batch_df.count()
    if n == 0:
        return
    feats = batch_df.na.drop(subset=["quantity", "price", "revenue"])
    assembler = VectorAssembler(inputCols=["quantity", "price", "revenue"], outputCol="raw_features")
    scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
    vec = assembler.transform(feats)
    vec = scaler.fit(vec).transform(vec)            # standardise so no single feature dominates
    model = KMeans(k=3, seed=42, featuresCol="features").fit(vec)
    preds = model.transform(vec)
    print("\\n===== Micro-batch " + str(batch_id) + ": " + str(n) + " transactions clustered =====")
    (preds.groupBy("prediction")
          .agg(count("*").alias("transactions"),
               sround(avg("quantity"), 1).alias("avg_qty"),
               sround(avg("price"), 2).alias("avg_price"),
               sround(avg("revenue"), 2).alias("avg_revenue"))
          .orderBy("prediction").show())

query = (parsed.writeStream
         .foreachBatch(cluster_batch)
         .trigger(processingTime="5 seconds")
         .start())

query.awaitTermination(timeout=60)    # run ~60s, then stop
query.stop()
spark.stop()
print("Streaming clustering finished.")
'''
with open(r"C:\Users\kgeet\bbim707-streaming\scripts\stream_clustering.py", "w", newline="\n") as f:
    f.write(script)
print("Wrote stream_clustering.py")

Wrote stream_clustering.py
